In [ ]:
import pandas as pd

# === Lista para almacenar resultados ===
resultados = []

# === Aplicar Segmentación + Grad-CAM ===
id_counter = 1
for nombre_archivo in os.listdir(imagenes_path):
    try:
        img_path = os.path.join(imagenes_path, nombre_archivo)
        
        # 1. Segmentar imagen (fondo negro)
        img_segmentada = segmentar_pie_con_fondo_negro(img_path)
        img_rgb = cv2.cvtColor(img_segmentada, cv2.COLOR_BGR2RGB)
        img_array = tf.keras.preprocessing.image.img_to_array(img_rgb)
        img_orig = img_array.copy().astype(np.uint8)

        # 2. Preprocesar para MobileNetV2
        input_array = np.expand_dims(img_array, axis=0)
        input_array = tf.keras.applications.mobilenet_v2.preprocess_input(input_array)

        # 3. Grad-CAM
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(input_array)
            pred_index = tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]

        grads = tape.gradient(class_channel, conv_outputs)[0]
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1))

        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        heatmap = np.maximum(heatmap, 0)
        heatmap /= np.max(heatmap + 1e-8)

        # 4. Superponer el heatmap
        heatmap = cv2.resize(heatmap, IMG_SIZE)
        heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
        superimposed_img = cv2.addWeighted(img_orig, 0.6, heatmap_color, 0.4, 0)

        # 5. Marcar el punto más caliente
        (ymax, xmax) = np.unravel_index(np.argmax(heatmap), heatmap.shape)
        cv2.circle(superimposed_img, (xmax, ymax), 8, (255, 255, 255), 2)

        # 6. Guardar resultados en tabla
        confianza = float(tf.reduce_max(predictions[0]).numpy()) * 100
        resultados.append({
            "Id": id_counter,
            "Nombre_Archivo": nombre_archivo,
            "Clase_Predicha": int(pred_index.numpy()),
            "Confianza_%": round(confianza, 2)
        })

        id_counter += 1

        # 7. Mostrar resultado Grad-CAM (opcional)
        plt.figure(figsize=(6, 6))
        plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
        plt.title(f"{nombre_archivo}\nGrad-CAM (Segmentado)\nClase: {pred_index.numpy()}")
        plt.axis('off')
        plt.show()

    except Exception as e:
        print(f"❌ Error procesando {nombre_archivo}: {e}")

# === Crear y mostrar tabla en Jupyter ===
df_resultados = pd.DataFrame(resultados, columns=["Id", "Nombre_Archivo", "Clase_Predicha", "Confianza_%"])
display(df_resultados)   # Tabla 1: resultados individuales

# === Tabla resumen con conteo por clase ===
conteo_clases = df_resultados["Clase_Predicha"].value_counts().sort_index()
df_conteo = pd.DataFrame({
    "Clase": conteo_clases.index,
    "Total_Imagenes": conteo_clases.values
})
display(df_conteo)   # Tabla 2: resumen por clase
